# Унифицированная валидация симуляции NovaSeq PE150 — мышь (fastp q30/u40)

Проверка ветки `ERP003950` по набору истины `post_annotation_filtered`.
Все артефакты валидации записываются идемпотентно: файл заменяется только если его содержимое изменилось.

In [ ]:
import csv, gzip, hashlib, io, json, math, os, re, shutil, subprocess
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pysam

DATASET = "ERP003950"
BRANCH = "insilicoseq_150bp_novaseq_post_annotation_filtered_random_cut_amp_in10x_rb3x_ss250"

SAMPLES = [
    "ERR346596",
    "ERR346597",
    "ERR346598",
    "ERR346599",
    "ERR346600",
    "ERR346601",
]
RUN_LOCUS = {sample: "IGH" for sample in SAMPLES}

# Сопоставимо с проверкой human по общему объёму выравнивания:
# 6 x 50k = 300k пар simulated reads. Для полной проверки установите None.
VALIDATION_PAIRS_PER_SAMPLE = 50_000
PROFILE_PAIRS_PER_SAMPLE = 50_000
NPROC = 8
FORCE = False

def resolve_root():
    candidates = []
    if os.environ.get("BCR_VOLUME"):
        candidates.append(Path(os.environ["BCR_VOLUME"]))
    candidates += [
        Path("/data/user/epishkin"),
        Path("/Users/epishkin/workspace/bcr-assembler"),
    ]
    start = Path.cwd().resolve()
    candidates += [start, *start.parents]
    seen = set()
    for root in candidates:
        key = str(root)
        if key in seen:
            continue
        seen.add(key)
        if (root / "results" / DATASET / "simulated" / BRANCH).is_dir():
            return root
    raise FileNotFoundError(
        f"Не найден results/{DATASET}/simulated/{BRANCH}; задайте BCR_VOLUME."
    )

ROOT = resolve_root()
DATASET_DIR = ROOT / "results" / DATASET
SIM_DIR = DATASET_DIR / "simulated" / BRANCH

TRUTH_DIR = SIM_DIR / "00_primary_truth"
PCR1_DIR = SIM_DIR / "01_pcr1"
FRAG_DIR = SIM_DIR / "02_fragmentation"
PCR2_DIR = SIM_DIR / "03_pcr2"
ALLOC_DIR = SIM_DIR / "04_read_allocation"
FASTQ_DIR = SIM_DIR / "06_fastq_pe150"
SIM_QC_DIR = SIM_DIR / "qc"

POST_FILTER_DIR = DATASET_DIR / "post_annotation_filtered"
FILTERED_ASSEMBLED_DIR = POST_FILTER_DIR / "fastq"
FILTER_SUMMARY_PATH = POST_FILTER_DIR / "filter_summary.json"
PR_TRIMMED_DIR = DATASET_DIR / "pr_trimmed" / "fastq"

VALIDATION_DIR = SIM_DIR / "validation"
SUBSET_DIR = VALIDATION_DIR / "subset_fastq"
FRAG_ALIGN_DIR = VALIDATION_DIR / "fragment_alignment"
TEMPLATE_ALIGN_DIR = VALIDATION_DIR / "template_alignment"
for d in (VALIDATION_DIR, SUBSET_DIR, FRAG_ALIGN_DIR, TEMPLATE_ALIGN_DIR):
    d.mkdir(parents=True, exist_ok=True)

FINAL_QC = SIM_QC_DIR / "final_qc.tsv"
if not FINAL_QC.exists():
    raise FileNotFoundError(f"{FINAL_QC} отсутствует; сначала доведите симуляцию до раздела final QC.")

for tool in ("bowtie2", "bowtie2-build", "samtools"):
    if not shutil.which(tool):
        raise RuntimeError(f"{tool} не найден в PATH")

print("ROOT:", ROOT)
print("DATASET:", DATASET)
print("BRANCH:", BRANCH)
print("SIM_DIR:", SIM_DIR)
print("samples:", ", ".join(SAMPLES))

## 1. Унифицированная запись результатов: заменить только если содержимое изменилось

In [ ]:
def sha256_file(path, decompress_gzip=False, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    opener = gzip.open if decompress_gzip else open
    with opener(path, "rb") as src:
        while True:
            chunk = src.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def files_identical(a, b):
    a, b = Path(a), Path(b)
    if not a.exists() or not b.exists():
        return False
    if a.suffix == ".gz" and b.suffix == ".gz":
        return sha256_file(a, decompress_gzip=True) == sha256_file(b, decompress_gzip=True)
    return sha256_file(a) == sha256_file(b)

def promote_if_changed(tmp, final, label=None):
    tmp, final = Path(tmp), Path(final)
    label = label or final.name
    if final.exists() and files_identical(tmp, final):
        tmp.unlink()
        print(f"[identical -> keep] {label}")
        return False
    final.parent.mkdir(parents=True, exist_ok=True)
    tmp.replace(final)
    print(f"[updated] {label}")
    return True

def write_text_if_changed(path, text):
    path = Path(path)
    tmp = Path(str(path) + ".tmp")
    tmp.write_text(text)
    return promote_if_changed(tmp, path)

def write_json_if_changed(path, obj):
    return write_text_if_changed(
        path,
        json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True) + "\n",
    )

def write_tsv_if_changed(path, rows, fieldnames=None):
    path = Path(path)
    rows = list(rows)
    if not rows:
        raise ValueError(f"Нет строк для {path}")
    fieldnames = fieldnames or list(rows[0])
    tmp = Path(str(path) + ".tmp")
    with open(tmp, "w", newline="") as h:
        w = csv.DictWriter(h, fieldnames=fieldnames, delimiter="\t", lineterminator="\n")
        w.writeheader()
        w.writerows(rows)
    return promote_if_changed(tmp, path)

def run_to_log(cmd, log_path):
    log_path = Path(log_path)
    tmp = Path(str(log_path) + ".tmp")
    print("[run]", " ".join(map(str, cmd)), flush=True)
    with open(tmp, "w") as h:
        subprocess.run(list(map(str, cmd)), stdout=h, stderr=subprocess.STDOUT, check=True)
    promote_if_changed(tmp, log_path)

def read_fastq_record(h):
    a = h.readline()
    if not a:
        return None
    b, c, d = h.readline(), h.readline(), h.readline()
    if not d:
        raise RuntimeError("Обрезанный FASTQ")
    return a, b, c, d

def count_fastq(path):
    n, lengths = 0, set()
    with gzip.open(path, "rt") as h:
        while True:
            rec = read_fastq_record(h)
            if rec is None:
                break
            head, seq, plus, qual = rec
            seq = seq.rstrip("\r\n")
            qual = qual.rstrip("\r\n")
            if not head.startswith("@") or not plus.startswith("+") or len(seq) != len(qual):
                raise RuntimeError(f"Некорректный FASTQ: {path}")
            n += 1
            lengths.add(len(seq))
    return n, sorted(lengths)

## 2. Внутренние инварианты симуляции

In [ ]:
def read_tsv(path):
    with open(path, newline="") as h:
        return list(csv.DictReader(h, delimiter="\t"))

final_rows = read_tsv(FINAL_QC)
final_by_sample = {row["sample"]: row for row in final_rows}
missing = sorted(set(SAMPLES) - set(final_by_sample))
if missing:
    raise RuntimeError(f"Samples отсутствуют в final_qc.tsv: {missing}")

internal_rows = []
expected_pairs_by_sample = {}

for sample in SAMPLES:
    q = final_by_sample[sample]
    if str(q["valid"]).strip().lower() != "true":
        raise RuntimeError(f"{sample}: simulation final_qc.tsv is not valid")

    expected_pairs = int(q["expected_pairs"])
    expected_pairs_by_sample[sample] = expected_pairs

    r1 = FASTQ_DIR / f"{sample}_R1.fastq.gz"
    r2 = FASTQ_DIR / f"{sample}_R2.fastq.gz"
    pcr1_tsv = PCR1_DIR / f"{sample}_pcr1_pool.tsv"
    fragment_tsv = FRAG_DIR / f"{sample}_fragments.tsv"
    allocation_tsv = ALLOC_DIR / f"{sample}_allocation.tsv"
    fragment_fasta = ALLOC_DIR / f"{sample}_selected_fragments.fasta"
    template_fasta = TRUTH_DIR / f"{sample}_templates.fasta"

    for p in (r1, r2, pcr1_tsv, fragment_tsv, allocation_tsv, fragment_fasta, template_fasta):
        if not p.exists():
            raise FileNotFoundError(p)

    n1, l1 = count_fastq(r1)
    n2, l2 = count_fastq(r2)
    if n1 != n2 or n1 != expected_pairs:
        raise RuntimeError(f"{sample}: pair-count mismatch: expected={expected_pairs:,}, R1={n1:,}, R2={n2:,}")
    if l1 != [150] or l2 != [150]:
        raise RuntimeError(f"{sample}: ожидался PE150, получено R1={l1}, R2={l2}")

    pcr1 = read_tsv(pcr1_tsv)
    fragments = read_tsv(fragment_tsv)
    allocation = read_tsv(allocation_tsv)

    pcr1_copies = sum(int(r["pcr_copies"]) for r in pcr1 if int(r["pcr_copies"]) > 0)
    fragment_input = sum(int(r["fragment_input_copies"]) for r in fragments)
    if pcr1_copies != fragment_input:
        raise RuntimeError(f"{sample}: масса фрагментации не сохранена: PCR1={pcr1_copies:,}, fragment_input={fragment_input:,}")

    allocated_pairs = sum(int(r["simulated_read_pairs"]) for r in allocation)
    if allocated_pairs != expected_pairs:
        raise RuntimeError(f"{sample}: allocated={allocated_pairs:,} != expected={expected_pairs:,}")

    nonzero_alloc = [r for r in allocation if int(r["simulated_read_pairs"]) > 0]
    internal_rows.append({
        "sample": sample,
        "expected_pairs": expected_pairs,
        "R1_pairs": n1,
        "R2_pairs": n2,
        "read_length": l1[0],
        "PCR1_molecules": pcr1_copies,
        "fragment_input_molecules": fragment_input,
        "fragmentation_mass_conserved": pcr1_copies == fragment_input,
        "allocated_pairs": allocated_pairs,
        "selected_fragment_species": len(nonzero_alloc),
    })

display(pd.DataFrame(internal_rows))
write_tsv_if_changed(VALIDATION_DIR / "internal_invariants.tsv", internal_rows)

## 3. Детерминированная подвыборка simulated reads

In [ ]:
def write_deterministic_subset(r1, r2, out1, out2, total_pairs, target_pairs):
    if target_pairs is None or target_pairs >= total_pairs:
        return r1, r2, total_pairs

    stride = max(1, total_pairs // target_pairs)
    tmp1 = Path(str(out1) + ".tmp")
    tmp2 = Path(str(out2) + ".tmp")
    selected = 0

    with gzip.open(r1, "rt") as h1, gzip.open(r2, "rt") as h2, \
         gzip.open(tmp1, "wt") as o1, gzip.open(tmp2, "wt") as o2:
        i = 0
        while True:
            a, b = read_fastq_record(h1), read_fastq_record(h2)
            if a is None or b is None:
                if a is not None or b is not None:
                    raise RuntimeError("R1/R2 заканчиваются в разных местах")
                break
            if i % stride == 0 and selected < target_pairs:
                o1.writelines(a)
                o2.writelines(b)
                selected += 1
            i += 1

    if selected != target_pairs:
        raise RuntimeError(f"Выбрано {selected}, ожидалось {target_pairs}")

    promote_if_changed(tmp1, out1)
    promote_if_changed(tmp2, out2)
    return out1, out2, selected

validation_inputs = {}
for sample in SAMPLES:
    r1 = FASTQ_DIR / f"{sample}_R1.fastq.gz"
    r2 = FASTQ_DIR / f"{sample}_R2.fastq.gz"
    sub1 = SUBSET_DIR / f"{sample}_R1.validation.fastq.gz"
    sub2 = SUBSET_DIR / f"{sample}_R2.validation.fastq.gz"

    if FORCE or not (sub1.exists() and sub2.exists()):
        vr1, vr2, n = write_deterministic_subset(
            r1, r2, sub1, sub2,
            expected_pairs_by_sample[sample],
            VALIDATION_PAIRS_PER_SAMPLE,
        )
    else:
        vr1, vr2 = sub1, sub2
        n1, _ = count_fastq(sub1)
        n2, _ = count_fastq(sub2)
        if n1 != n2:
            raise RuntimeError(f"{sample}: кэшированные validation mates различаются")
        n = n1

    validation_inputs[sample] = {"R1": vr1, "R2": vr2, "pairs": n}
    print(sample, "validation pairs:", f"{n:,}")

total_validation_pairs = sum(v["pairs"] for v in validation_inputs.values())
print("total simulated validation pairs:", f"{total_validation_pairs:,}")

## 4. Выравнивание simulated reads на точные фрагменты и V–J шаблоны (per-sample)

In [ ]:
def ensure_index(reference, prefix):
    reference = Path(reference)
    prefix = Path(prefix)
    fingerprint = prefix.parent / f"{prefix.name}.reference.sha256"
    current_hash = sha256_file(reference)
    marker = Path(str(prefix) + ".1.bt2")
    marker_l = Path(str(prefix) + ".1.bt2l")
    index_exists = marker.exists() or marker_l.exists()
    fingerprint_ok = fingerprint.exists() and fingerprint.read_text().strip() == current_hash

    if index_exists and fingerprint_ok:
        print("[index unchanged]", prefix)
        return prefix

    for p in prefix.parent.glob(prefix.name + "*.bt2*"):
        p.unlink()
    run_to_log(
        ["bowtie2-build", "--threads", str(NPROC), reference, prefix],
        prefix.parent / "bowtie2_build.log",
    )
    write_text_if_changed(fingerprint, current_hash + "\n")
    return prefix

def align_pair(reference, outdir, label, r1, r2):
    outdir = Path(outdir)
    outdir.mkdir(parents=True, exist_ok=True)
    index = ensure_index(reference, outdir / "index")

    tmp_sam = outdir / f".{label}.new.sam"
    tmp_bam = outdir / f".{label}.new.bam"
    tmp_bai = Path(str(tmp_bam) + ".bai")
    final_bam = outdir / f"{label}.bam"
    final_bai = Path(str(final_bam) + ".bai")

    run_to_log([
        "bowtie2", "--very-sensitive-local", "-p", str(NPROC),
        "-x", index, "-1", r1, "-2", r2, "-S", tmp_sam
    ], outdir / f"{label}.bowtie2_align.log")

    subprocess.run(["samtools", "sort", "-@", str(NPROC), "-o", str(tmp_bam), str(tmp_sam)], check=True)
    subprocess.run(["samtools", "index", str(tmp_bam)], check=True)
    tmp_sam.unlink(missing_ok=True)

    if final_bam.exists() and files_identical(tmp_bam, final_bam):
        tmp_bam.unlink()
        tmp_bai.unlink(missing_ok=True)
        print(f"[identical -> keep] {final_bam.name}")
    else:
        tmp_bam.replace(final_bam)
        tmp_bai.replace(final_bai)
        print(f"[updated] {final_bam.name}")

    return final_bam

fragment_bams = {}
template_bams = {}
for sample in SAMPLES:
    fragment_bams[sample] = align_pair(
        ALLOC_DIR / f"{sample}_selected_fragments.fasta",
        FRAG_ALIGN_DIR / sample,
        "selected_fragments",
        validation_inputs[sample]["R1"],
        validation_inputs[sample]["R2"],
    )
    template_bams[sample] = align_pair(
        TRUTH_DIR / f"{sample}_templates.fasta",
        TEMPLATE_ALIGN_DIR / sample,
        "primary_templates",
        validation_inputs[sample]["R1"],
        validation_inputs[sample]["R2"],
    )
print("per-sample alignments done")

In [ ]:
def flagstat_metrics(bam):
    txt = subprocess.run(
        ["samtools", "flagstat", str(bam)],
        capture_output=True, text=True, check=True
    ).stdout
    mapped_pct = properly_paired_pct = None
    for line in txt.splitlines():
        if " mapped (" in line and "primary mapped" not in line:
            m = re.search(r"\(([\d.]+)%", line)
            if m:
                mapped_pct = float(m.group(1))
        if " properly paired (" in line:
            m = re.search(r"\(([\d.]+)%", line)
            if m:
                properly_paired_pct = float(m.group(1))
    return mapped_pct, properly_paired_pct

def samtools_error_rate(bam):
    txt = subprocess.run(
        ["samtools", "stats", str(bam)],
        capture_output=True, text=True, check=True
    ).stdout
    for line in txt.splitlines():
        if "error rate:" in line:
            parts = line.split("\t")
            for i, x in enumerate(parts):
                if x.strip() == "error rate:" and i + 1 < len(parts):
                    return float(parts[i + 1])
    return None

alignment_rows = []
for sample in SAMPLES:
    frag_mapped, frag_proper = flagstat_metrics(fragment_bams[sample])
    frag_error = samtools_error_rate(fragment_bams[sample])
    tpl_mapped, tpl_proper = flagstat_metrics(template_bams[sample])
    tpl_error = samtools_error_rate(template_bams[sample])
    alignment_rows.append({
        "sample": sample,
        "frag_mapped_pct": frag_mapped,
        "frag_properly_paired_pct": frag_proper,
        "frag_error_rate": frag_error,
        "template_mapped_pct": tpl_mapped,
        "template_properly_paired_pct": tpl_proper,
        "template_error_rate": tpl_error,
    })

display(pd.DataFrame(alignment_rows))
write_tsv_if_changed(VALIDATION_DIR / "alignment_metrics.tsv", alignment_rows)

## 5. Соответствие simulated read породившему его фрагменту

In [ ]:
def iter_fasta(path):
    with open(path) as h:
        name, chunks = None, []
        for line in h:
            line = line.rstrip("\r\n")
            if line.startswith(">"):
                if name is not None:
                    yield name, "".join(chunks)
                name, chunks = line[1:].split()[0], []
            else:
                chunks.append(line)
        if name is not None:
            yield name, "".join(chunks)

QNAME_RE = re.compile(r"^(.*_frag\d+)_\d+_\d+(?:/[12])?$")

def expected_fragment_id(qname):
    m = QNAME_RE.match(qname)
    return m.group(1) if m else None

def fragment_origin_metrics(fragment_fasta, bam, max_primary_reads=500_000):
    fragment_sequences = dict(iter_fasta(fragment_fasta))
    parsed = exact = equivalent = primary = 0
    with pysam.AlignmentFile(str(bam), "rb") as h:
        for r in h.fetch(until_eof=True):
            if r.is_unmapped or r.is_secondary or r.is_supplementary:
                continue
            primary += 1
            exp = expected_fragment_id(r.query_name)
            if exp is not None and exp in fragment_sequences:
                parsed += 1
                if exp == r.reference_name:
                    exact += 1
                    equivalent += 1
                elif (
                    r.reference_name in fragment_sequences
                    and fragment_sequences[exp] == fragment_sequences[r.reference_name]
                ):
                    equivalent += 1
            if primary >= max_primary_reads:
                break
    return {
        "primary_reads_checked": primary,
        "qname_origin_parse_rate": parsed / primary if primary else None,
        "exact_fragment_origin_rate": exact / parsed if parsed else None,
        "sequence_equivalent_fragment_origin_rate": equivalent / parsed if parsed else None,
    }

fragment_origin_rows = []
for sample in SAMPLES:
    m = fragment_origin_metrics(
        ALLOC_DIR / f"{sample}_selected_fragments.fasta",
        fragment_bams[sample],
    )
    fragment_origin_rows.append({"sample": sample, **m})
    print(sample, m)

display(pd.DataFrame(fragment_origin_rows))
write_tsv_if_changed(VALIDATION_DIR / "fragment_origin.tsv", fragment_origin_rows)

## 6. Matched-подвыборка реальных paired-end reads

`post_annotation_filtered` содержит уже собранные pRESTO-последовательности, поэтому их нельзя напрямую сравнивать с simulated PE150. Используем идентификаторы собранных последовательностей, прошедших post-annotation filter, и возвращаемся к соответствующим парам в `pr_trimmed/fastq`.

In [ ]:
FILTER_SUMMARY = json.loads(FILTER_SUMMARY_PATH.read_text())

def normalized_coord(header):
    x = header.strip()
    if x.startswith("@"):
        x = x[1:]
    x = x.split(None, 1)[0]
    x = x.split("|", 1)[0]
    x = re.sub(r"/[12]$", "", x)
    return x

def proportional_quotas(total):
    denom = sum(expected_pairs_by_sample.values())
    exact = {s: total * expected_pairs_by_sample[s] / denom for s in SAMPLES}
    q = {s: int(math.floor(v)) for s, v in exact.items()}
    remainder = total - sum(q.values())
    for s in sorted(SAMPLES, key=lambda x: exact[x] - q[x], reverse=True)[:remainder]:
        q[s] += 1
    return q

REAL_QUOTAS = proportional_quotas(total_validation_pairs)

def select_filtered_coordinates(sample, target):
    fq = FILTERED_ASSEMBLED_DIR / f"{sample}_filtered.fastq.gz"
    total = int(FILTER_SUMMARY["samples"][sample]["passed"])
    stride = max(1, total // target)
    selected = []
    with gzip.open(fq, "rt") as h:
        i = 0
        while True:
            rec = read_fastq_record(h)
            if rec is None:
                break
            if i % stride == 0 and len(selected) < target:
                selected.append(normalized_coord(rec[0]))
            i += 1
    if len(selected) != target:
        raise RuntimeError(f"{sample}: выбрано {len(selected)} вместо {target}")
    return set(selected)

selected_coordinates = {
    sample: select_filtered_coordinates(sample, REAL_QUOTAS[sample])
    for sample in SAMPLES
}
print({sample: len(ids) for sample, ids in selected_coordinates.items()})

In [ ]:
REAL_R1 = SUBSET_DIR / "ERP003950_real_postfilter_R1.validation.fastq.gz"
REAL_R2 = SUBSET_DIR / "ERP003950_real_postfilter_R2.validation.fastq.gz"

def build_real_matched_subset():
    tmp1 = Path(str(REAL_R1) + ".tmp")
    tmp2 = Path(str(REAL_R2) + ".tmp")
    found = Counter()

    with gzip.open(tmp1, "wt") as o1, gzip.open(tmp2, "wt") as o2:
        for sample in SAMPLES:
            wanted = selected_coordinates[sample]
            r1 = PR_TRIMMED_DIR / f"{sample}_1.pr.fastq.gz"
            r2 = PR_TRIMMED_DIR / f"{sample}_2.pr.fastq.gz"
            with gzip.open(r1, "rt") as h1, gzip.open(r2, "rt") as h2:
                while True:
                    a, b = read_fastq_record(h1), read_fastq_record(h2)
                    if a is None or b is None:
                        if a is not None or b is not None:
                            raise RuntimeError(f"{sample}: R1/R2 имеют разную длину")
                        break
                    ca, cb = normalized_coord(a[0]), normalized_coord(b[0])
                    if ca != cb:
                        raise RuntimeError(f"{sample}: несинхронная пара {ca} != {cb}")
                    if ca in wanted:
                        o1.writelines(a)
                        o2.writelines(b)
                        found[sample] += 1

    for sample in SAMPLES:
        if found[sample] != REAL_QUOTAS[sample]:
            raise RuntimeError(
                f"{sample}: найдено {found[sample]} matched-пар, ожидалось {REAL_QUOTAS[sample]}. "
                "Проверьте сохранение SRA coordinate между pr_trimmed и assembled."
            )

    promote_if_changed(tmp1, REAL_R1)
    promote_if_changed(tmp2, REAL_R2)
    return dict(found)

real_subset_counts = build_real_matched_subset()
print(real_subset_counts)

## 7. Real и simulated на одном объединённом V–J референсе

In [ ]:
# Объединяем per-sample simulated subsets и шаблоны в единый референс,
# чтобы real и simulated выравнивались на одинаковый V-J truth.

COMBINED_SIM_R1 = SUBSET_DIR / "ERP003950_simulated_all_R1.validation.fastq.gz"
COMBINED_SIM_R2 = SUBSET_DIR / "ERP003950_simulated_all_R2.validation.fastq.gz"
COMBINED_TEMPLATES = TRUTH_DIR / "combined_primary_templates.fasta"

def concat_files(paths, out):
    tmp = Path(str(out) + ".tmp")
    with open(tmp, "wb") as o:
        for p in paths:
            with open(p, "rb") as src:
                shutil.copyfileobj(src, o, 1024 * 1024)
    promote_if_changed(tmp, out)

concat_files([validation_inputs[s]["R1"] for s in SAMPLES], COMBINED_SIM_R1)
concat_files([validation_inputs[s]["R2"] for s in SAMPLES], COMBINED_SIM_R2)

def concat_fasta(paths, out):
    tmp = Path(str(out) + ".tmp")
    with open(tmp, "w") as o:
        for p in paths:
            with open(p) as src:
                shutil.copyfileobj(src, o)
    promote_if_changed(tmp, out)

concat_fasta([TRUTH_DIR / f"{s}_templates.fasta" for s in SAMPLES], COMBINED_TEMPLATES)

REAL_TEMPLATE_BAM = align_pair(
    COMBINED_TEMPLATES,
    TEMPLATE_ALIGN_DIR,
    "real_postfilter_primary_templates",
    REAL_R1,
    REAL_R2,
)
SIM_TEMPLATE_BAM = align_pair(
    COMBINED_TEMPLATES,
    TEMPLATE_ALIGN_DIR,
    "simulated_primary_templates",
    COMBINED_SIM_R1,
    COMBINED_SIM_R2,
)

def softclip_bases(read):
    if not read.cigartuples:
        return 0
    return sum(length for op, length in read.cigartuples if op == 4)

def bam_read_metrics(bam, max_primary_reads=500_000):
    rows = []
    with pysam.AlignmentFile(str(bam), "rb") as h:
        for r in h.fetch(until_eof=True):
            if r.is_secondary or r.is_supplementary or r.is_unmapped:
                continue
            qlen = r.query_length or 0
            nm = r.get_tag("NM") if r.has_tag("NM") else np.nan
            locus = None
            if r.reference_name:
                m = re.search(r"_(IGH|IGK|IGL)_tpl_", r.reference_name)
                locus = m.group(1) if m else None
            rows.append({
                "mapq": r.mapping_quality,
                "nm_per_base": (nm / qlen) if qlen and not pd.isna(nm) else np.nan,
                "softclip_fraction": softclip_bases(r) / qlen if qlen else np.nan,
                "insert_size": (
                    abs(r.template_length)
                    if r.is_read1 and r.is_proper_pair and r.template_length
                    else np.nan
                ),
                "locus": locus,
            })
            if len(rows) >= max_primary_reads:
                break
    return pd.DataFrame(rows)

real_reads = bam_read_metrics(REAL_TEMPLATE_BAM)
sim_reads = bam_read_metrics(SIM_TEMPLATE_BAM)

def alignment_summary_row(label, bam, df):
    mapped, proper = flagstat_metrics(bam)
    return {
        "dataset": label,
        "mapped_pct": mapped,
        "properly_paired_pct": proper,
        "error_rate": samtools_error_rate(bam),
        "median_mapq": float(df["mapq"].median()),
        "mean_nm_per_base": float(df["nm_per_base"].mean()),
        "mean_softclip_fraction": float(df["softclip_fraction"].mean()),
        "median_insert_size": float(df["insert_size"].dropna().median()),
        "primary_mapped_reads_profiled": int(len(df)),
    }

alignment_rows_real_sim = [
    alignment_summary_row("real_postfilter", REAL_TEMPLATE_BAM, real_reads),
    alignment_summary_row("simulated", SIM_TEMPLATE_BAM, sim_reads),
]
write_tsv_if_changed(VALIDATION_DIR / "real_vs_simulated_alignment.tsv", alignment_rows_real_sim)
display(pd.DataFrame(alignment_rows_real_sim))

## 8. GC, качество оснований и повторяемость последовательностей

In [ ]:
def pair_profiles(r1, r2, max_pairs, max_cycle=150):
    qsum1 = np.zeros(max_cycle, dtype=float)
    qsum2 = np.zeros(max_cycle, dtype=float)
    qn1 = np.zeros(max_cycle, dtype=int)
    qn2 = np.zeros(max_cycle, dtype=int)
    gc = []
    pair_hashes = Counter()
    pairs = 0

    with gzip.open(r1, "rt") as h1, gzip.open(r2, "rt") as h2:
        while pairs < max_pairs:
            a, b = read_fastq_record(h1), read_fastq_record(h2)
            if a is None or b is None:
                break
            s1, q1 = a[1].strip().upper(), a[3].strip()
            s2, q2 = b[1].strip().upper(), b[3].strip()

            for arr, cnt, q in ((qsum1, qn1, q1), (qsum2, qn2, q2)):
                for i, ch in enumerate(q[:max_cycle]):
                    arr[i] += ord(ch) - 33
                    cnt[i] += 1

            seq = s1 + s2
            if seq:
                gc.append(100 * (seq.count("G") + seq.count("C")) / len(seq))
            pair_hashes[hashlib.sha1((s1 + "|" + s2).encode()).hexdigest()] += 1
            pairs += 1

    return {
        "pairs": pairs,
        "q1": np.divide(qsum1, qn1, out=np.full(max_cycle, np.nan), where=qn1 > 0),
        "q2": np.divide(qsum2, qn2, out=np.full(max_cycle, np.nan), where=qn2 > 0),
        "gc": np.asarray(gc),
        "duplicate_pair_fraction": (
            sum(v for v in pair_hashes.values() if v > 1) / pairs if pairs else np.nan
        ),
    }

real_prof = pair_profiles(REAL_R1, REAL_R2, PROFILE_PAIRS_PER_SAMPLE)
sim_prof = pair_profiles(COMBINED_SIM_R1, COMBINED_SIM_R2, PROFILE_PAIRS_PER_SAMPLE)

profile_rows = [
    {
        "dataset": "real_postfilter",
        "pairs": real_prof["pairs"],
        "mean_gc_pct": float(np.mean(real_prof["gc"])),
        "median_gc_pct": float(np.median(real_prof["gc"])),
        "duplicate_pair_fraction": float(real_prof["duplicate_pair_fraction"]),
        "mean_q_R1": float(np.nanmean(real_prof["q1"])),
        "mean_q_R2": float(np.nanmean(real_prof["q2"])),
    },
    {
        "dataset": "simulated",
        "pairs": sim_prof["pairs"],
        "mean_gc_pct": float(np.mean(sim_prof["gc"])),
        "median_gc_pct": float(np.median(sim_prof["gc"])),
        "duplicate_pair_fraction": float(sim_prof["duplicate_pair_fraction"]),
        "mean_q_R1": float(np.nanmean(sim_prof["q1"])),
        "mean_q_R2": float(np.nanmean(sim_prof["q2"])),
    },
]
write_tsv_if_changed(VALIDATION_DIR / "real_vs_simulated_sequence_qc.tsv", profile_rows)
display(pd.DataFrame(profile_rows))

In [ ]:
def save_figure_if_changed(fig, path):
    path = Path(path)
    tmp = Path(str(path) + ".tmp.png")
    fig.savefig(tmp, dpi=160, bbox_inches="tight")
    promote_if_changed(tmp, path)

cycles = np.arange(1, 151)

fig = plt.figure(figsize=(10, 5))
plt.plot(cycles, real_prof["q1"], label="Real R1")
plt.plot(cycles, sim_prof["q1"], label="Simulated R1")
plt.xlabel("Позиция в read")
plt.ylabel("Средний Phred")
plt.title("R1: качество оснований")
plt.legend()
save_figure_if_changed(fig, VALIDATION_DIR / "quality_profile_R1.png")
plt.show()

fig = plt.figure(figsize=(10, 5))
plt.plot(cycles, real_prof["q2"], label="Real R2")
plt.plot(cycles, sim_prof["q2"], label="Simulated R2")
plt.xlabel("Позиция в read")
plt.ylabel("Средний Phred")
plt.title("R2: качество оснований")
plt.legend()
save_figure_if_changed(fig, VALIDATION_DIR / "quality_profile_R2.png")
plt.show()

fig = plt.figure(figsize=(9, 5))
plt.hist(real_prof["gc"], bins=60, density=True, alpha=0.5, label="Real")
plt.hist(sim_prof["gc"], bins=60, density=True, alpha=0.5, label="Simulated")
plt.xlabel("GC, %")
plt.ylabel("Плотность")
plt.title("GC-состав")
plt.legend()
save_figure_if_changed(fig, VALIDATION_DIR / "gc_distribution.png")
plt.show()

## 9. Состав локусов по результатам общего V–J выравнивания

In [ ]:
def locus_rows(df, label):
    counts = df["locus"].dropna().value_counts()
    total = counts.sum()
    return [
        {
            "dataset": label,
            "locus": locus,
            "mapped_reads": int(n),
            "pct": float(100 * n / total) if total else np.nan,
        }
        for locus, n in counts.items()
    ]

locus_rows_all = (
    locus_rows(real_reads, "real_postfilter")
    + locus_rows(sim_reads, "simulated")
)
write_tsv_if_changed(
    VALIDATION_DIR / "real_vs_simulated_locus_alignment.tsv",
    locus_rows_all,
)
display(pd.DataFrame(locus_rows_all))

## 10. Единая итоговая сводка

`validation_summary.json` — главный итоговый файл: внутренние инварианты, per-sample alignment-метрики, fragment origin и блок `realism_comparison` (real-vs-simulated).

In [ ]:
summary = {
    "dataset": DATASET,
    "branch": BRANCH,
    "samples": SAMPLES,
    "validation_pairs_per_sample": {
        s: validation_inputs[s]["pairs"] for s in SAMPLES
    },
    "internal_invariants": internal_rows,
    "alignment_metrics": alignment_rows,
    "fragment_origin": fragment_origin_rows,
    "realism_comparison": {
        "real_subset_pairs": int(sum(real_subset_counts.values())),
        "real_subset_pairs_by_sample": real_subset_counts,
        "alignment": {
            row["dataset"]: {k: v for k, v in row.items() if k != "dataset"}
            for row in alignment_rows_real_sim
        },
        "sequence_qc": {
            row["dataset"]: {k: v for k, v in row.items() if k != "dataset"}
            for row in profile_rows
        },
        "locus_alignment": locus_rows_all,
        "interpretation": (
            "Внутреннее выравнивание simulated->truth проверяет корректность симуляции; "
            "real-vs-simulated на одном объединённом V-J референсе проверяет сходство "
            "поведения reads. Полная оценка пригодности для benchmark BCR reconstruction "
            "дополнительно требует запуска TRUST4 и сравнения реконструкции с известным truth."
        ),
    },
}

write_json_if_changed(VALIDATION_DIR / "validation_summary.json", summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))

## Интерпретация

Сильная валидация требует одновременно:

1. **Simulated → truth**: высокая доля mapping и properly paired, низкая частота ошибок, сохранение инвариантов масс PCR1 → фрагментация → allocation и бюджета ридов per-sample.
2. **Real ↔ simulated**: сопоставимые alignment-метрики на одном объединённом V–J референсе (MAPQ, NM, soft clipping, insert size), сопоставимый GC-профиль и качество оснований, отсутствие радикального расхождения по повторяемости.

Все 6 samples мыши — IGH, поэтому проверка состава цепей тривиальна; основной акцент — per-sample инварианты и real-vs-simulated сравнение. 100% mapping simulated reads само по себе не означает реалистичности: real-vs-simulated сравнение включено в этот же ноутбук.